# 开放车间调度问题 (OSSP)

**类别：** 调度

来源：[https://www.hexaly.com/templates/open-shop-scheduling-problem-ossp](https://www.hexaly.com/templates/open-shop-scheduling-problem-ossp)


## 问题描述

**在开放车间调度问题 (OSSP)** 中，一组作业必须在车间内的每台机器上进行处理。每个作业由一组无序的任务（称为活动）组成。一个活动表示该作业在一台机器上的处理过程，具有给定的处理时间。每个作业在每台机器上都有一个活动，并且当该作业的另一个活动仍在运行时，不能开始新的活动。每台机器一次只能处理一个活动。目标是找到一个使 makespan（所有作业处理完成的时间）最小的作业排序方案。

	

### 建模要点

- 添加 [区间决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/intervalvariables.html) 来建模活动
- 添加 [list 决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 来建模每个作业和每台机器上的活动顺序
- 定义 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 将区间变量和 list 变量联系起来


## 数据

我们提供的开放车间调度问题 (OSSP) 实例来自 [Taillard](http://mistic.heig-vd.ch/taillard/problemes.dir/ordonnancement.dir/ordonnancement.html)。数据文件的格式如下：

- 第一行：作业数、机器数、用于生成实例的种子、之前找到的上界和下界
- 对每个作业：每个活动在其指定机器上的处理时间
- 对每个作业：分配给每个活动的机器 ID。


## 模型

开放车间调度问题 (OSSP) 的 Hexaly 模型使用区间决策变量来表示活动的时间范围。区间的长度由每个活动的处理时间约束。

除了区间变量外，我们还使用 [list 决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html)。与 [Job Shop](https://www.hexaly.com/example/job-shop-scheduling-problem-jssp) 示例类似，list 对机器上或作业内的活动进行排序。通过使用 **count** 算子约束 list 的大小，我们确保每个作业在每台机器上都被处理。

析取资源约束（即每台机器一次只能处理一个活动）可以重新表述如下：对所有 i，在位置 i+1 处理的活动必须在其位置 i 处理的活动结束后才开始。为了建模这些约束，我们将区间决策（时间范围）与 list 决策（作业排序）配对。我们编写一个 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 来表达两个连续活动之间的关系。该函数用于每台机器所处理的所有活动的可变参数 **and** 算子中。

我们使用相同的策略来建模活动的析取约束。对所有作业和所有 i，作业中位置 i+1 的活动必须在该作业中位置 i 的活动结束后才开始。与析取资源约束类似，我们使用一个 lambda 函数结合可变参数 **and** 算子来建模这些约束，覆盖构成每个作业的所有活动。

目标是最小化 makespan，即所有活动处理完成的时间。


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys


def read_instance(filename):
    # The input files follow the "Taillard" format
    with open(filename, 'r') as f:
        lines = f.readlines()

    first_line = lines[1].split()
    nb_jobs = int(first_line[0])
    nb_machines = int(first_line[1])

    # Processing times for each job on each machine
    # (given in the task order, the processing order is a decision variable)
    processing_times_task_order = [[int(proc_time) for proc_time in line.split()]
                                   for line in lines[3:3 + nb_jobs]]

    # Index of machines for each task
    machine_index = [[int(machine_i) - 1 for machine_i in line.split()]
                     for line in lines[4 + nb_jobs:4 + 2 * nb_jobs]]

    # Reorder processing times: processingTime[j][m] is the processing time of the
    # task of job j that is processed on machine m
    processing_times = [[processing_times_task_order[j][machine_index[j].index(m)]
                         for m in range(nb_machines)] for j in range(nb_jobs)]

    # Trivial upper bound for the end time of tasks
    max_end = sum(map(lambda processing_times_job: sum(processing_times_job), processing_times))

    return nb_jobs, nb_machines, processing_times, max_end


def main(instance_file, output_file, time_limit):
    nb_jobs, nb_machines, processing_times, max_end = read_instance(instance_file)

    with hexaly.optimizer.HexalyOptimizer() as optimizer:
        #
        # Declare the optimization model
        #
        model = optimizer.model

        # Interval decisions: time range of each task
        # tasks[j][m] is the interval of time of the task of job j
        # which is processed on machine m
        tasks = [[model.interval(0, max_end) for _ in range(nb_machines)] for _ in range(nb_jobs)]

        # Task duration constraints
        for j in range(nb_jobs):
            for m in range(0, nb_machines):
                model.constraint(model.length(tasks[j][m]) == processing_times[j][m])

        # Create an Hexaly array in order to be able to access it with "at" operators
        task_array = model.array(tasks)

        # List of the jobs on each machine
        jobs_order = [model.list(nb_jobs) for _ in range(nb_machines)]
        for m in range(nb_machines):
            # Each job is scheduled on every machine
            model.constraint(model.eq(model.count(jobs_order[m]), nb_jobs))

            # Every machine executes a single task at a time
            sequence_lambda = model.lambda_function(lambda i:
                model.at(task_array, jobs_order[m][i], m) < model.at(task_array, jobs_order[m][i + 1], m))
            model.constraint(model.and_(model.range(0, nb_jobs - 1), sequence_lambda))

        # List of the machines for each job
        machines_order = [model.list(nb_machines) for _ in range(nb_jobs)]
        for j in range(nb_jobs):
            # Every task is scheduled on its corresponding machine
            model.constraint(model.eq(model.count(machines_order[j]), nb_machines))

            # A job has a single task at a time
            sequence_lambda = model.lambda_function(lambda k:
                    model.at(task_array, j, machines_order[j][k]) < model.at(task_array, j, machines_order[j][k + 1]))
            model.constraint(model.and_(model.range(0, nb_machines - 1), sequence_lambda))

        # Minimize the makespan: the end of the last task
        makespan = model.max([model.end(model.at(task_array, j, m))
                             for j in range(nb_jobs) for m in range(nb_machines)])
        model.minimize(makespan)

        model.close()

        # Parametrize the optimizer
        optimizer.param.time_limit = time_limit
        optimizer.solve()

        #
        # Write the solution in a file with the following format:
        #  - for each machine, the job sequence
        #
        if output_file is not None:
            with open(output_file, 'w') as f:
                for m in range(nb_machines):
                    line = ""
                    for j in range(nb_jobs):
                        line += str(jobs_order[m].value[j]) + " "
                    f.write(line + "\n")
            print("Solution written in file ", output_file)


if __name__ == '__main__':
    if len(sys.argv) < 2:
        print(
            "Usage: python openshop.py instance_file [output_file] [time_limit]")
        sys.exit(1)

    instance_file = sys.argv[1]
    output_file = sys.argv[2] if len(sys.argv) >= 3 else None
    time_limit = int(sys.argv[3]) if len(sys.argv) >= 4 else 60
    main(instance_file, output_file, time_limit)
